<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/12-case-studies-and-capstone/01-customer-support-assistant.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 案例研究 A —— 客服助手：从需求界定到上线（以及凌晨两点的报警）

**目标：** 用*一个*真实场景把本仓库的大部分内容串起来：一句含糊的客户诉求，经过界定、度量、构建、上线，然后在**生产环境中排障**，关键节点都有可运行的代码。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **怎么读这篇。** 叙述本身就是案例；代码单元格是每个阶段的*一个极小可运行切片*（6 篇文章的语料、5 个问题的黄金集），小到几秒就能跑完，又真实到能展示机制，尤其是第 8 阶段的回归。完整技术都在各阶段链接到的章节里。这就是 [capstone](CAPSTONE.zh.md) 的形态；照着建你自己的项目之前先读它。

## 准备

自包含。接下来两个单元格会安装本仓库的 `aien` 辅助包（它会引入 `groq` 客户端），以及用于 embedding 的 `sentence-transformers`，并加载你的 key。

在 [console.groq.com](https://console.groq.com/) 领免费 key；在 Colab 里通过**钥匙图标** → 名称为 `GROQ_API_KEY` 的 secret 添加。（完整走查：[00-setup](../00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" sentence-transformers

In [ ]:
from aien import setup
client, MODEL = setup()   # 加载 GROQ_API_KEY，返回一个可用的 Groq 客户端和默认模型

## 诉求（你实际接到的东西）

你是一家厂商的 FDE。一家中型 SaaS 公司 **Northwind** 签了约。第一次通话，他们的支持副总裁说：

> *「我们团队快淹了。新产品上线后工单涨了 3 倍，招人根本跟不上。你们能不能用 AI 搞点什么，让客户更快拿到答案？」*

就这些。没有规格说明，没有数据集，没有成功指标。**这种含糊就是这份工作本身**，不是客户的失误；而从这句话到一个已部署系统之间的那段距离，就是整个案例研究。

> **🚩 常见错误 ——** 听到「用 AI 搞点什么」就立刻去抓一个框架和向量数据库。你还不知道「更快的答案」指什么、什么算*对*、它值多少钱。构建顺序是：**界定 → 度量 → 构建 → 上线 → 运营**。先伸手去写代码，就是你快速交付错误东西的方式。

## 第 1 阶段 —— 把模糊的诉求界定清楚 → [11 客户手艺](../11-customer-craft/01-scoping-and-discovery.zh.ipynb)

在写任何代码之前，先跑一遍第 11 节的 discovery 问题，把「用 AI 搞点什么」变成一个有范围、*可评测*的系统。

会改变设计的问题：
- **用户是谁，客户还是客服人员？** Northwind 的回答：*客服人员*。这是一个决定，不是一个细节。面向客服人员的「起草回复」工具有人在环路里（风险更低，第 07 节），UX 也和面向客户的机器人不同。你界定的范围是客服辅助版本。
- **答案*从哪来*？** 他们的帮助中心（约 1,200 篇文章）+ 已解决的历史工单。这就是你的语料，**真实而混乱**（重复、过时文章、只存在于工单里的部落知识）。
- **什么算「对」？** 不是「听起来不错」，而是*一位资深客服会发出的答案，且有真实文章作为依据*。这个定义就是你的评测。
- **它值多少钱？** 他们的数字：缩短中位处理时长，拦截前 20 个重复问题。这给了你目标和预算。

一页纸的**范围界定文档**（第 11 节的交付物）由此而来：用户、问题、范围内/范围外、成功指标，以及你两周后要展示的 demo。

> **🔵 面试信号 ——** 在任何架构之前先问「用户是谁、什么算正确答案？」，这就是资深的表现。它也是让下游一切可度量的原因。

## 第 2 阶段 —— 在动手构建*之前*先装上指标 → [02 评测 I](../02-evals-basics/01-measuring-outputs.zh.ipynb)

第 02 节的整篇论点：**在你构建那个将来要调优的东西之前，先给「它好不好」安上一个数字。** 所以下一步——在检索之前、在任何模型调用之前——是一个极小的**黄金集**：真实问题，配上一位资深客服会发出的答案以及它来自哪篇文章。

下面是我们的迷你 Northwind：一个 6 篇文章的帮助中心和一个 5 个问题的黄金集。（真实版本约有 1,200 篇文章和约 40 个问题；机制完全一样。）

In [ ]:
# 语料：一个极小的帮助中心。每篇文章有 id、标题和正文。
CORPUS = {
    "acct-reset":   "Reset your password: go to Settings > Security > Reset password. A link is emailed to you.",
    "acct-email":   "Change your email: Settings > Account > Email > Edit. Confirm via the link we send.",
    "billing-refund":"Refunds are available within 30 days of purchase. Reply to your receipt email with your order id.",
    "api-status":   "Check live API status at status.northwind.io. Incidents are posted there first.",
    "api-key-err":  "Error NW-403 means an invalid API key. Regenerate the key under Settings > Developer > API keys.",
    "export-data":  "Export your data: Settings > Data > Export. You receive a CSV by email within an hour.",
}

# 黄金集：question ->（应该回答它的文章、资深客服会发出的要点）。
GOLDEN = [
    {"q": "How do I reset my password?",        "article": "acct-reset",    "gist": "Settings > Security > Reset password; link emailed."},
    {"q": "Can I get a refund?",                 "article": "billing-refund","gist": "Within 30 days; reply to receipt with order id."},
    {"q": "I'm getting error NW-403 on the API", "article": "api-key-err",   "gist": "NW-403 = bad API key; regenerate under Settings > Developer."},
    {"q": "How do I change my email address?",   "article": "acct-email",    "gist": "Settings > Account > Email > Edit; confirm via link."},
    {"q": "How do I get my data out?",           "article": "export-data",   "gist": "Settings > Data > Export; CSV emailed within an hour."},
]
print(f"{len(CORPUS)} articles, {len(GOLDEN)} golden questions — the yardstick every later decision is measured against.")

> **⚠️ 生产现实 ——** 团队会跳过这一步，因为感觉像没在构建。然后他们「改进」了一周 prompt，却完全不知道它有没有变好。黄金集就是把那一周变成一张图的东西。

## 第 3 阶段 —— 构建检索 → [03 RAG](../03-rag/00-what-is-rag.zh.ipynb)

现在是 RAG 系统，按第 03 节教的顺序：**检索优先**（[03/01](../03-rag/01-embeddings-retrieval.zh.ipynb)）：给文章做 embedding，做向量检索。之后你会加混合检索 + rerank（[03/02](../03-rag/02-hybrid-and-reranking.zh.ipynb)），并回头重看分块（[03/03](../03-rag/03-chunking.zh.ipynb)）。这里我们构建向量检索核心，以及——关键的——一个可以重跑的 `index_corpus()` 步骤，这在第 8 阶段很重要。

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # 小、CPU 友好

# INDEX 是一个独立产物，从语料构建而来。故意做成可重跑：
# RAG 系统的正确性取决于这个索引与语料保持同步。
def index_corpus(corpus):
    ids = list(corpus)
    vecs = embedder.encode([corpus[i] for i in ids], normalize_embeddings=True)
    return {"ids": ids, "vecs": vecs}

def retrieve(query, index, k=1):
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    scores = index["vecs"] @ qv                 # 余弦（向量已归一化）
    top = np.argsort(scores)[::-1][:k]
    return [index["ids"][i] for i in top]

INDEX = index_corpus(CORPUS)

# 健全性检查：在黄金集上做检索。注意它可能会漏掉术语化的 NW-403 那条 ——
# 这正是 03/01 讲的词汇不匹配陷阱，混合检索 + BM25 能修好它。
hits = sum(retrieve(g["q"], INDEX)[0] == g["article"] for g in GOLDEN)
print(f"retrieval hit rate: {hits}/{len(GOLDEN)}")
for g in GOLDEN:
    got = retrieve(g["q"], INDEX)[0]
    print(f"  {'OK ' if got==g['article'] else 'MISS'} {g['q'][:38]:40} -> {got}")

> **🚩 常见错误 ——** 把「答案烂」当成生成问题，然后去抓更大的模型。第 [03/04（RAG 为什么会失败）](../03-rag/04-why-rag-fails.zh.ipynb) 节说得很明确：几乎总是**检索**的问题。你的黄金集会告诉你到底是哪个：一个错误答案，如果它的正确文章根本没被检索到，那就是检索 bug，不是模型 bug。

## 第 4 阶段 —— 把它做成助手 → [05 Agent](../05-agents/01-agent-loop-from-scratch.zh.ipynb)

检索负责找到文章；助手*根据它们起草回复*。Northwind 的工单有时需要一次查询（「#4021 订单发货了吗？」），所以设计给模型一个工具（[05/02](../05-agents/02-tool-design.zh.ipynb)），但保留一道**人工闸门**（[05/03](../05-agents/03-guardrails-and-budgets.zh.ipynb)）：它起草，客服人员批准。第 1 阶段的那个范围决策，现在成了一条安全属性。

这里我们只保留核心：检索文章，然后让模型起草一份有依据的回复。（完整的有界循环 + `order_status` 工具是第 05 节的活儿；这里只有评测要打分的作答路径。）

In [ ]:
def draft_reply(question, index):
    """检索最相关的文章，然后起草一份有依据的回复。发送前由人工批准。"""
    art_id = retrieve(question, index)[0]
    article = CORPUS[art_id]
    msgs = [
        {"role": "system", "content": "You are a support assistant. Answer ONLY from the provided article. "
                                       "Be terse. If the article doesn't cover it, say so."},
        {"role": "user", "content": f"Article:\n{article}\n\nCustomer question: {question}"},
    ]
    resp = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=120, temperature=0)
    return {"article": art_id, "reply": resp.choices[0].message.content.strip()}

demo = draft_reply("How do I reset my password?", INDEX)
print("retrieved:", demo["article"])
print("draft reply:", demo["reply"])

> **💡 为什么不用更大的 agent ——** Northwind 的大多数问题都是「找到文章并把答案说出来」。这几乎算不上一个 agent。把循环保持得小一些；只在任务真正分叉的地方（查订单）才伸手去用工具。第 [05/03](../05-agents/03-guardrails-and-budgets.zh.ipynb) 节「pipeline 胜过 agent」的告诫，在这个构建内部同样适用。*（案例研究 B 把那个判断一路推到底——什么时候干脆不要 agent。）*

## 第 5 阶段 —— 差异点：给答案质量打分 → [04 评测 II](../04-evals/01-golden-sets.zh.ipynb)

第 3 阶段度量的是*检索*。现在度量**答案质量**，也就是 Northwind 花钱买的东西，用 **LLM-as-judge**（[04/02](../04-evals/02-llm-as-judge.zh.ipynb)）：起草的回复是否匹配资深客服的要点？把它包成一个 `eval_quality()` 分数，这样一个改动就变成一个数字，而不是一场争论。这个分数接进 CI（[04/03](../04-evals/03-regression-evals.zh.ipynb)），就是第 8 阶段里救你的东西。

In [ ]:
import re

def judge(question, reply, gist):
    """LLM-as-judge：这条回复传达的解决方案是否和资深客服的要点一致？1/0。"""
    msgs = [
        {"role": "system", "content": "You grade support replies. Output ONLY '1' if the reply conveys the "
                                       "same core resolution as the reference, else '0'. No other text."},
        {"role": "user", "content": f"Question: {question}\nReference: {gist}\nReply: {reply}\nGrade (1 or 0):"},
    ]
    out = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=2, temperature=0)
    return 1 if "1" in (out.choices[0].message.content or "") else 0

def eval_quality(index, verbose=False):
    """整个系统端到端地在黄金集上跑一遍 -> 一个 [0,1] 区间的质量分。"""
    passed = 0
    for g in GOLDEN:
        d = draft_reply(g["q"], index)
        ok = judge(g["q"], d["reply"], g["gist"])
        passed += ok
        if verbose:
            print(f"  {'PASS' if ok else 'FAIL'} [{d['article']}] {g['q'][:36]}")
    return passed / len(GOLDEN)

baseline = eval_quality(INDEX, verbose=True)
print(f"\nBASELINE quality score: {baseline:.0%}")

> **🔵 面试信号 ——**「我们在 CI 里有一道 LLM-judge 评测闸门，并用人工标注抽查过」这句话，区分开了*做过 demo* 的工程师和*交付过系统*的工程师。它也是「你怎么知道它是好的？」这个问题的诚实答案。

## 第 6 阶段 —— 上线与容量估算 → [09 服务化](../09-serving-inference/01-serving-frameworks.zh.ipynb) + [10 系统设计](../10-ml-system-design/01-designing-an-inference-service.zh.ipynb)

Northwind 有约 200 名客服，工作时间内流量有突发。用第 10 节的方法做容量估算。没有新代码，只有面试官期待的那套算术：

- **托管 endpoint 还是自托管？** 在这个量级下，**托管 API**（本仓库使用的 OpenAI 兼容接缝）比自建 GPU 更便宜、更省事，这也是 [09/01](../09-serving-inference/01-serving-frameworks.zh.ipynb) 的默认选择。记下在 vLLM 上自托管这条路，等量级大到值得时再用。
- **那些数字**（[10/01](../10-ml-system-design/01-designing-an-inference-service.zh.ipynb)）：峰值并发客服 → QPS → tokens/s → 每月成本，对重复的 system 开场白用 prompt 缓存。

In [ ]:
# 来自 10/01 的粗略估算 —— 回答「要多少 GPU / 账单多少？」
def size_it(peak_concurrent_agents, reqs_per_agent_per_min, tokens_per_req, tok_per_s_per_gpu=2500, util=0.7):
    import math
    qps = peak_concurrent_agents * reqs_per_agent_per_min / 60
    demand_tok_s = qps * tokens_per_req
    gpus = math.ceil(demand_tok_s / (tok_per_s_per_gpu * util))
    return {"qps": round(qps, 1), "demand_tok_s": round(demand_tok_s), "gpus_if_self_hosted": gpus}

print(size_it(peak_concurrent_agents=200, reqs_per_agent_per_min=2, tokens_per_req=300))
print("-> at this scale, a managed endpoint is the call; self-host on vLLM when the bill crosses GPU break-even.")

> **⚠️ 生产现实 ——** 面试官会追问「10 倍量级呢？」因为做过容量估算，你回答的是一个调节手段，而不是一次重新设计：更多副本，或者在托管账单跨过 GPU 盈亏平衡点时自托管到 vLLM。

## 第 7 阶段 —— 交付 demo → 回到 [11 客户手艺](../11-customer-craft/01-scoping-and-discovery.zh.ipynb)

两周后，你在 Northwind 面前做 demo，**用他们的真实工单、现场跑**，而不是一个精心挑选的脚本（第 11 节的 demo 纪律）。你展示评测数字，而不是感觉：*「在黄金集上，N% 的回复匹配了资深客服的回复，且有真实文章作为依据；这些是它答错的，以及原因。」* 诚实的失败案例比一场完美的照稿演示更能建立信任。

上线时你接上**可观测性**（[08/01](../08-operations/01-observability-and-llmops.zh.ipynb)）：追踪每一次调用，安全地记录 prompt（不含客户 PII，这与 [07](../07-security/01-prompt-injection-and-trust.zh.ipynb) 相关），跟踪成本/延迟/错误率。它向一个试点小组上线。

## 第 8 阶段 —— 两周后，凌晨两点的报警（构建 → **排障**）

系统本来运行正常。现在试点客服反馈：**「答案开始不对劲了。」** 质量下滑，没人部署过模型变更，而 CI 里的评测闸门是**绿的**。这就是这份工作的另一半，**诊断一次线上回归**，也正是你构建的可观测性和评测收回成本的地方。

我们*可以真的复现它*。Northwind 的内容团队做了一次帮助中心清理，**重写并重命名了文章**，但没有人重跑 `index_corpus()`。索引现在指向旧文本。看着分数掉下去，再恢复。

In [ ]:
# 内容团队的迁移：文章被重写/扩写（id 不变，措辞与结构变了）。
CORPUS_V2 = dict(CORPUS)
CORPUS_V2["acct-reset"]  = ("Forgot or resetting credentials? Navigate to your profile, open the Security "
                            "panel, and choose 'Reset password'. We dispatch a one-time link to your inbox.")
CORPUS_V2["api-key-err"] = ("Authentication failures (formerly NW-403) are now surfaced as AUTH_INVALID_KEY. "
                            "Rotate credentials from the Developer console under API credentials.")
CORPUS_V2["billing-refund"] = ("Our refund window is one calendar month from the transaction date. Initiate "
                               "from the Billing portal, or reply to the emailed receipt with your reference id.")

# BUG 所在：语料变了，但索引没有重建。检索仍在对旧向量打分。
CORPUS.update(CORPUS_V2)          # 生产内容现在是 V2……
# ……但 INDEX 是过期的（还是第 3 阶段那批 V1 向量）。

regressed = eval_quality(INDEX, verbose=True)
print(f"\nPRODUCTION quality score now: {regressed:.0%}   (baseline was {baseline:.0%})")
print("CI is green because CI runs against a FROZEN copy — it never saw the migration.")

In [ ]:
# 修复：给迁移后的语料重新建索引。第 3 阶段搭的那个接缝，重跑只需一行。
INDEX = index_corpus(CORPUS)      # 用当前（V2）内容重建向量
recovered = eval_quality(INDEX)
print(f"after re-indexing: {recovered:.0%}   (was {regressed:.0%} while stale)")
print("\nDurable fix: add index_corpus() to the content-publishing workflow, plus a daily CANARY eval")
print("that runs the golden set against PRODUCTION — so the next drift pages a dashboard, not an agent.")

**诊断过程，写成可复用的决策树**（这是技能，不是那个具体的 bug）：

1. **用一个数字复现，而不是凭感觉。** 对着*生产环境*跑 [04 回归评测](../04-evals/03-regression-evals.zh.ipynb)，而不是那份冻结的 CI 副本。分数确实掉了。
2. **是检索还是生成？**（[03/04 的视角](../03-rag/04-why-rag-fails.zh.ipynb)）拉出 trace（[08/01](../08-operations/01-observability-and-llmops.zh.ipynb)）：模型*基于它检索到的内容*回复得没问题，但检索返回的是差一点的近似结果。**是检索的问题。**
3. **如果不是代码变了，那是什么变了？** **数据**移动了：文章在上游被重写。索引**过期了**。
4. **根因：** 迁移之后没有人重新建索引，一个无人负责的**数据新鲜度依赖**。

> **🚩 常见错误 ——** 认定质量回归意味着「模型变差了」或「有人改了 prompt」。在 RAG 系统里，最常见的原因是**数据动了而索引没动**——一道绿色的 CI 闸门抓不到它从未见过的漂移。构建它是接上索引；调试它，是知道要首先怀疑数据边界。

> **🔵 面试信号 ——**「我们的 CI 评测是绿的，但生产质量掉了，因为语料在上游被重新索引而我们这边的向量过期了——所以我们加了一道生产 canary 评测」是一个*资深*的实战故事。它表明你能把离线评测与生产监控分开，并且以数据依赖而非仅仅代码来思考。

## 这个案例演练了什么

一个场景，本仓库的大部分内容，以及这份工作的两半：

| 阶段 | 章节 | 技能 |
|---|---|---|
| 界定诉求 | [11](../11-customer-craft/01-scoping-and-discovery.zh.ipynb) | discovery、范围界定文档 |
| 指标优先 | [02](../02-evals-basics/01-measuring-outputs.zh.ipynb) | 构建前先有黄金集 |
| 检索 | [03](../03-rag/00-what-is-rag.zh.ipynb) | RAG、混合检索、分块、失败诊断 |
| 助手 | [05](../05-agents/01-agent-loop-from-scratch.zh.ipynb) | agent 循环、工具、护栏 |
| 质量闸门 | [04](../04-evals/01-golden-sets.zh.ipynb) | LLM-judge、把回归评测接进 CI |
| 上线与容量估算 | [09](../09-serving-inference/01-serving-frameworks.zh.ipynb)、[10](../10-ml-system-design/01-designing-an-inference-service.zh.ipynb) | 托管 vs 自托管、容量算术 |
| 安全 | [07](../07-security/01-prompt-injection-and-trust.zh.ipynb) | 不可信工单、安全日志、人工闸门 |
| 上线 + demo | [11](../11-customer-craft/01-scoping-and-discovery.zh.ipynb)、[08](../08-operations/01-observability-and-llmops.zh.ipynb) | 诚实的 demo、trace |
| **凌晨两点的报警** | [08](../08-operations/01-observability-and-llmops.zh.ipynb)、[04](../04-evals/03-regression-evals.zh.ipynb)、[03/04](../03-rag/04-why-rag-fails.zh.ipynb) | **诊断一次线上回归** |

> **⭐ 关键收获 ——** 你从来不缺那些零件；你在第 00–11 节把它们一个个搭了出来。真实项目就是在客户的约束下*组合*它们，诚实地交付，并且——大多数作品集都跳过的另一半——**让它在上线之后继续正常工作。** 把这条弧线讲清楚，就是你能带进面试的最有力的东西。现在去建你自己的：[capstone](CAPSTONE.zh.md)。